In [3]:
from PIL import Image

def convert_to_ico(
    input_image_path: str,
    output_ico_path: str
):
    # ICO에 포함할 표준 아이콘 크기들
    icon_sizes = [
        (16, 16),
        (32, 32),
        (48, 48),
        (64, 64),
        (128, 128),
        (256, 256),
    ]

    img = Image.open(input_image_path)

    # 투명 배경 유지 (PNG → ICO 대응)
    if img.mode != "RGBA":
        img = img.convert("RGBA")

    img.save(
        output_ico_path,
        format="ICO",
        sizes=icon_sizes
    )

    print(f"ICO file saved to: {output_ico_path}")


convert_to_ico(
    input_image_path="menu_icon.png",   # 1000x1000 원본
    output_ico_path="menu_icon.ico"
)

ICO file saved to: menu_icon.ico


In [3]:
import pickle
from pathlib import Path

pkl_path = Path("./app/data/laser/feature_columns_v7.pkl")

print(f"PKL exists: {pkl_path.exists()}")
print(f"Absolute path: {pkl_path.resolve()}")

with open(pkl_path, "rb") as f:
    obj = pickle.load(f)

print("\n=== BASIC INFO ===")
print("type:", type(obj))

try:
    print("len:", len(obj))
except Exception as e:
    print("len: ERROR ->", e)

print("\n=== CONTENT PREVIEW ===")
if isinstance(obj, (list, tuple)):
    print("all:", obj[:])
elif hasattr(obj, "head"):
    print(obj.head())
else:
    print(obj)

print("\n=== ELEMENT TYPE CHECK (first 10) ===")
if isinstance(obj, (list, tuple)):
    for i, v in enumerate(obj[:]):
        print(f"[{i}] value={v}, type={type(v)}")

print("\n=== ALL ELEMENT TYPES (unique) ===")
if isinstance(obj, (list, tuple)):
    unique_types = {type(v) for v in obj}
    for t in unique_types:
        print(t)

PKL exists: True
Absolute path: C:\Users\ty\Documents\Shooting\api\app\data\laser\feature_columns_v7.pkl

=== BASIC INFO ===
type: <class 'list'>
len: 46

=== CONTENT PREVIEW ===
all: ['point_x_1', 'point_x_2', 'point_x_3', 'point_x_4', 'point_x_5', 'point_x_6', 'point_x_7', 'point_x_8', 'point_x_9', 'point_x_10', 'point_y_1', 'point_y_2', 'point_y_3', 'point_y_4', 'point_y_5', 'point_y_6', 'point_y_7', 'point_y_8', 'point_y_9', 'point_y_10', 'score_1', 'score_2', 'score_3', 'score_4', 'score_5', 'score_6', 'score_7', 'score_8', 'score_9', 'score_10', 'ttf_1', 'ttf_2', 'ttf_3', 'ttf_4', 'ttf_5', 'ttf_6', 'ttf_7', 'ttf_8', 'ttf_9', 'ttf_10', 'coi_x', 'coi_y', 'mr', 'std_x', 'std_y', 'total_ttf']

=== ELEMENT TYPE CHECK (first 10) ===
[0] value=point_x_1, type=<class 'str'>
[1] value=point_x_2, type=<class 'str'>
[2] value=point_x_3, type=<class 'str'>
[3] value=point_x_4, type=<class 'str'>
[4] value=point_x_5, type=<class 'str'>
[5] value=point_x_6, type=<class 'str'>
[6] value=point_x

In [1]:
import sys

import traceback
import pandas as pd
from tqdm import tqdm

from app.routers.shootinganalysis import (
    load_all_game_records,
    load_game_record,
    calculate_result,
)
from app.models.schemas import ShootingResult

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# -------------------------------------------------
# Batch runner
# -------------------------------------------------

def run_batch_analysis() -> pd.DataFrame:
    GAME_RECORD_SP = load_all_game_records()
    session_ids = GAME_RECORD_SP["hd_id"].dropna().unique()
    print(f"Total sessions to process: {len(session_ids)}")

    results = []

    for game_id in tqdm(session_ids, desc="Processing sessions"):
        try:
            df = load_game_record(str(game_id))

            shooting_result = [
                ShootingResult(
                    nth=int(row["nth"]),
                    score=float(row["score"]),
                    time=float(row["shot_time"]),
                    pointX=float(row["point_x"]),
                    pointY=float(row["point_y"]),
                    distance=int(row["distance"]),
                    color=str(row["color"]),
                )
                for _, row in df.iterrows()
            ]
            if len(shooting_result) != 10:
                continue
            coi, mean_radius, std, skill_level, threshold = calculate_result(shooting_result)

            results.append({
                "game_id": game_id,
                "shooting_result": shooting_result,
                "num_shots": len(shooting_result),
                "coi": coi,
                "mean_radius": mean_radius,
                "std": std,
                "threshold": threshold,
                "skill_level": skill_level,
            })

        except Exception as e:
            results.append({
                "game_id": game_id,
                "error": str(e),
                "traceback": traceback.format_exc(),
            })

    return pd.DataFrame(results)

In [2]:
df_result = run_batch_analysis()

# CSV 저장
df_result.to_csv("batch_analysis_result.csv", index=False, encoding="utf-8-sig")

Total sessions to process: 1285


Processing sessions:   4%|▍         | 50/1285 [00:00<00:02, 495.33it/s]

valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 1
skill_le

Processing sessions:  25%|██▌       | 323/1285 [00:00<00:01, 620.17it/s]

valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 3
skill_le

...

Processing sessions:  88%|████████▊ | 1125/1285 [00:02<00:00, 528.07it/s]

valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 1
skill_le

Processing sessions:  96%|█████████▋| 1240/1285 [00:02<00:00, 547.33it/s]

valid points len: 1
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 0
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 3
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 2
skill_level: 입문
valid points len: 1
skill_level: 입문
valid points len: 1
skill_le

Processing sessions: 100%|██████████| 1285/1285 [00:02<00:00, 548.30it/s]


In [3]:
df_result["skill_level"].value_counts()

skill_level
입문    769
Name: count, dtype: int64

In [ ]:
df_sampled = (
    df_result
    .groupby("skill_level", as_index=False)
    .head(5)
)

df_sampled[["skill_level", "game_id"]]

,skill_level,game_id
0,입문,20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601
1,입문,20250101-3d449ecb-535c-4e01-b23d-506653a83c87
2,입문,20250101-5d03cabc-b54e-44aa-abf8-9df335ebf14b
3,입문,20250101-86d8a0f8-2152-4029-888d-dbfdc2384ec3
4,입문,20250101-af59ade2-b543-4148-ae5e-fc6f2d67fb38


: 

In [1]:
import pandas as pd
from sqlalchemy import text

from app.utils.db import get_engine

def load_game_record(game_id: str) -> pd.DataFrame:
    """
    Load shooting records for a specific game session from MySQL.
    """
    engine = get_engine()

    query = text("""
        SELECT
            hd_id, 
            nth,
            score,
            point_x,
            point_y,
            distance,
            color,
            shot_time
        FROM game_record_dt
        WHERE hd_id = :game_id
        ORDER BY nth
    """)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"game_id": game_id})

    if df.empty:
        raise ValueError(f"Session not found: {game_id}")

    return df.reset_index(drop=True)

In [ ]:
game_id = "20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601"

df = load_game_record(game_id)

print(df)
print(len(df))

                                           hd_id  nth  score   point_x  \
0  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    1    9.2  0.414513   
1  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    2    9.9  0.514495   
2  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    3    9.8  0.460161   
3  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    4    8.6  0.464366   
4  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    5    9.7  0.475167   
5  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    6    9.5  0.514668   
6  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    7   10.2  0.503330   
7  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    8    8.4  0.486663   
8  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601    9    9.7  0.441468   
9  20250101-dc477ea7-f3a5-42b2-89a1-a492eff04601   10    9.5  0.514668   

    point_y  distance   color  shot_time  
0  0.522470      2598     red        1.4  
1  0.448585      2597     red        6.2  
2  0.460672      2592     red        8.4  
3  0.385265  

: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def calculate_confidence(std_ratio):
    lo, hi = 0.7, 1.3
    res = {"수직 분산": 0, "수평 분산": 0, "산란": 0}
    
    if std_ratio <= lo:
        dist = math.log(lo / std_ratio)
        res["수직 분산"] = 0.6 + 0.4 * math.tanh(3.0 * dist)
    elif std_ratio >= hi:
        dist = math.log(std_ratio / hi)
        res["수평 분산"] = 0.6 + 0.4 * math.tanh(3.0 * dist)
    else:
        if std_ratio <= 1.0:
            t = math.log(1.0 / std_ratio) / math.log(1.0 / lo)
        else:
            t = math.log(std_ratio) / math.log(hi)
        res["산란"] = 1.0 - 0.4 * t
    return res

# 데이터 생성 (0.1 ~ 3.0 범위)
r_values = np.linspace(0.1, 3.0, 500)
y_vert = [calculate_confidence(r)["수직 분산"] for r in r_values]
y_horz = [calculate_confidence(r)["수평 분산"] for r in r_values]
y_scat = [calculate_confidence(r)["산란"] for r in r_values]

plt.figure(figsize=(10, 6))
plt.plot(r_values, y_vert, label='Vertical Dispersion (Vertical)', color='blue', lw=2)
plt.plot(r_values, y_horz, label='Horizontal Dispersion (Horizontal)', color='red', lw=2)
plt.plot(r_values, y_scat, label='Scattering (Central)', color='green', lw=2)

# 경계선 표시
plt.axvline(x=0.7, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=1.3, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=1.0, color='black', linestyle=':', alpha=0.3)

plt.title("Shooting Error Confidence by Std Ratio", fontsize=14)
plt.xlabel("Std Ratio (σy / σx)", fontsize=12)
plt.ylabel("Confidence", fontsize=12)
plt.ylim(0, 1.1)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [2]:
inference_time = 13.5052
fps = 1.0 / (inference_time + 1e-6)
fps

0.0740455473413539